In [ ]:
import requests
import os 
from dotenv import load_dotenv
import pandas as pd 
import requests
from requests.exceptions import JSONDecodeError
import json


In [ ]:


url = "https://api.esios.ree.es/indicators"
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("ESIOS_API_KEY").strip().strip("'")

headers = {

    "Accept" :"application/json; application/vnd.esios-api-v1+json",
    "Content-Type" : "application/json",
    "x-api-key": API_KEY
}

response= requests.get(headers= headers, url =url)
data = response.json()


In [ ]:
data

In [ ]:
df = pd.DataFrame(data['indicators'])
df.shape

In [ ]:
indicadores_prediccion = {
    "demanda_prevista": 544,
    "demanda_prevista_diaria": 460,
    "eolica_prevista": 541,
    "eolica_prevista_d1": 1777,
    "solar_fv_prevista": 542,
    "solar_termica_prevista": 543
}

indicadores_explicativo = {
    "demanda_real": 1293,
    "eolica_real": 551,
    "solar_fv_real": 2044,
    "solar_termica_real": 2045,
    "ciclo_combinado_real": 550,
    "nuclear_real": 549,
    "hidraulica_real": 546,
    "intercambios_real": 553
}


In [ ]:
diccionario = {}
lista_errores = []
for nombre, id_indicador in indicadores_prediccion.items():
    url=f"https://api.esios.ree.es/indicators/{id_indicador}"
    headers = {

    "Accept" :"application/json; application/vnd.esios-api-v1+json",
    "Content-Type" : "application/json",
    "x-api-key": API_KEY
}
    params = {
        "start_date":"2023-01-01T00:00:00",
        "end_date":"2026-06-30T00:00:00",
        "time_trunc":"hour"
}   
    
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            data = response.json()
            diccionario[nombre] = data
            with open(rf"..\data\raw\esios\esios_datos_{nombre}_raw.json", "w") as archivo:
             json.dump(data, archivo)
            print(response.status_code)
            print(response.text)
        else:
            print(f"Error al obtener datos para {nombre}: {response.status_code} - {response.text}")
            lista_errores.append((nombre, response.status_code, response.text))
            
    except requests.exceptions.ConnectionError as e:
        print(f"Error de conexión para {nombre}: {e}")
        lista_errores.append((nombre, "ConnectionError", str(e)))
        
        continue
    except JSONDecodeError as e:
        print(f"Error al decodificar JSON para {nombre}: {e} con codigo de estado {response.status_code}")
        lista_errores.append((nombre, response.status_code, response.text))
        continue
        

In [ ]:
len(diccionario['demanda_prevista']['indicator']['values'])

In [ ]:
len(lista_errores)

In [ ]:
diccionario

In [ ]:
comprobacion = diccionario['demanda_prevista']['indicator']['values']

df_comprobacion = pd.DataFrame(comprobacion)

df_comprobacion['datetime'] = pd.to_datetime(df_comprobacion['datetime'], utc=True).dt.tz_localize(None)

rango_completo = pd.date_range(start='2023-01-01', end='2026-06-30', freq='1h')
datetimes_presentes = set(df_comprobacion['datetime'])
fechas_faltantes = [f for f in rango_completo if f not in datetimes_presentes]

print(len(fechas_faltantes))
fechas_faltantes

<-- ### Nota: verificación de cobertura temporal — ESIOS (demanda_prevista, id 544)

Al descargar el indicador `demanda_prevista` (id 544) para el rango completo 
2023-01-01 a 2026-06-30 con `time_trunc=hour`, se obtuvieron **30.622 registros**.

**Verificación:** se generó el rango horario teórico completo con `pd.date_range(freq='1h')` 
y se comparó contra las fechas presentes en la respuesta de la API (tras normalizar con 
`pd.to_datetime(..., utc=True)` para evitar el choque de offsets `+01:00`/`+02:00` propio 
de los cambios de horario). Se encontraron **4 horas "faltantes"**:

| Fecha | Causa |
|---|---|
| 2024-10-27 01:00:00 | Cambio de horario de invierno (última hora ambigua de octubre) |
| 2025-10-26 01:00:00 | Cambio de horario de invierno (última hora ambigua de octubre) |
| 2026-06-29 23:00:00 | Artefacto de borde en la conversión UTC ↔ hora local, en el límite final del rango solicitado |
| 2026-06-30 00:00:00 | Artefacto de borde en la conversión UTC ↔ hora local, en el límite final del rango solicitado |

**Interpretación:** 2 de los 4 huecos corresponden al fenómeno esperado y conocido del 
cambio de horario de otoño en España (hora local ambigua/duplicada). Los otros 2 son un 
efecto de los límites (`start_date`/`end_date`) de esta verificación puntual, no de la 
ingesta en sí.

**Decisión:** dado que representa un 0,013% del total de registros (4 de 30.622), 
no se considera necesario un tratamiento especial en esta fase. Se documenta como 
constancia de verificación de calidad de datos, siguiendo el mismo criterio aplicado a AEMET. -->


In [ ]:
for nombre in diccionario:
    print(f" {nombre} tiene {len(diccionario[nombre]['indicator']['values'])} valores")

Verificación de cobertura temporal — indicadores de predicción (ESIOS)

Se realizó un análisis exhaustivo de cobertura temporal (huecos, rachas y causas) sobre demanda_prevista, por ser la variable objetivo del modelo principal. Se detectaron 4 registros faltantes sobre un total teórico de ~30.648 (0,013%), causados por el cambio de horario de octubre (DST) y un artefacto de conversión UTC↔local en el borde final del rango de fechas. No se encontró ningún patrón de fallo estructural adicional.

Para el resto de indicadores del conjunto de predicción (demanda_prevista_diaria, eolica_prevista, eolica_prevista_d1, solar_fv_prevista, solar_termica_prevista) se verificó el número total de registros obtenidos, encontrando valores consistentes con el mismo orden de magnitud (entre 30.621 y 30.624). Esto sugiere que están sujetos al mismo patrón de micro-huecos por DST/borde de rango, sin indicios de problemas de cobertura distintos o más graves. No se realizó un análisis de rachas variable por variable, priorizando el esfuerzo de investigación según el impacto de cada variable en el modelo.

In [ ]:
diccionario_explicativo = {}
lista_errores_explicativo = []
for nombre, id_indicador in indicadores_explicativo.items():
    url=f"https://api.esios.ree.es/indicators/{id_indicador}"
    headers = {

    "Accept" :"application/json; application/vnd.esios-api-v1+json",
    "Content-Type" : "application/json",
    "x-api-key": API_KEY
}
    params = {
        "start_date":"2023-01-01T00:00:00",
        "end_date":"2026-06-30T00:00:00",
        "time_trunc":"hour"
}   
    
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            data_explicativo = response.json()
            diccionario_explicativo[nombre] = data_explicativo
            with open(rf"..\data\raw\esios\esios_datos_{nombre}_raw.json", "w") as archivo:
                         json.dump(data_explicativo, archivo)
            print(response.status_code)
            print(response.text)
        else:
            print(f"Error al obtener datos para {nombre}: {response.status_code} - {response.text}")
            lista_errores_explicativo.append((nombre, response.status_code, response.text))
            
    except requests.exceptions.ConnectionError as e:
        print(f"Error de conexión para {nombre}: {e}")
        lista_errores_explicativo.append((nombre, "ConnectionError", str(e)))
        
        continue
    except JSONDecodeError as e:
        print(f"Error al decodificar JSON para {nombre}: {e} con codigo de estado {response.status_code}")
        lista_errores_explicativo.append((nombre, response.status_code, response.text))
        continue

In [ ]:
len(lista_errores_explicativo)
lista_errores_explicativo


In [ ]:
diccionario_explicativo

In [ ]:
for nombre in diccionario_explicativo:
    print(f" {nombre} tiene {len(diccionario_explicativo[nombre]['indicator']['values'])} valores") 

In [ ]:


df_solar['datetime'] = pd.to_datetime(df_solar['datetime'], utc =True).dt.tz_localize(None) 

rango_completo = pd.date_range(start='2024-06-10', end='2026-06-30', freq='1H')
datetimes_presentes = set(df_solar['datetime'])
fechas_faltantes_solar = [f for f in rango_completo if f not in datetimes_presentes]

print(len(fechas_faltantes_solar))
fechas_faltantes_solar 

In [ ]:
fechas_faltantes_ordenadas = sorted(fechas_faltantes_solar)

# Mira las primeras y últimas para hacerte una idea rápida
print(fechas_faltantes_ordenadas[:5])
print(fechas_faltantes_ordenadas[-5:])

In [ ]:
diferencias = pd.Series(fechas_faltantes_ordenadas).diff()
diferencias.value_counts()

In [ ]:
# Índices donde la diferencia es mayor a 1 día (ahí empieza una racha nueva)
saltos = diferencias[diferencias > pd.Timedelta(hours=1)].index

# Usamos esos índices para trocear fechas_faltantes_ordenadas en sub-listas
inicio_racha = 0
for salto in list(saltos) + [len(fechas_faltantes_ordenadas)]:
    racha = fechas_faltantes_ordenadas[inicio_racha:salto]
    print(f"Racha de {len(racha)} horas: {racha[0]} → {racha[-1]}")
    inicio_racha = salto

In [ ]:
df[df['name'].str.contains('solar', na=False)]



In [ ]:
url_prueba = "https://api.esios.ree.es/indicators/10205"
headers = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "x-api-key": API_KEY
}
params = {
    "start_date": "2026-06-01T00:00:00",
    "end_date": "2026-06-30T00:00:00",
    "time_trunc": "hour"
}

response_prueba = requests.get(url_prueba, headers=headers, params=params)
print(response_prueba.status_code)
data_prueba = response_prueba.json()
print(len(data_prueba['indicator']['values']))
data_prueba['indicator']['values'][:3]

In [ ]:
set([v['geo_name'] for v in data_prueba['indicator']['values']])

In [ ]:
print(response_prueba.status_code)
print(response_prueba.text)

In [ ]:
url_prueba_2044 = "https://api.esios.ree.es/indicators/2044"
headers = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "x-api-key": API_KEY
}
params = {
    "start_date": "2023-01-01T00:00:00",
    "end_date": "2023-02-01T00:00:00",
    "time_trunc": "hour"
}

response_2044 = requests.get(url_prueba_2044, headers=headers, params=params)
print(response_2044.status_code)
data_2044 = response_2044.json()
print(len(data_2044['indicator']['values']))
print(set([v['geo_name'] for v in data_2044['indicator']['values']]))
data_2044['indicator']['values'][:3]

In [ ]:
url_prueba_2045 = "https://api.esios.ree.es/indicators/2044"
headers = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "x-api-key": API_KEY
}
params = {
    "start_date": "2023-01-01T00:00:00",
    "end_date": "2023-02-01T00:00:00",
    "time_trunc": "hour"
}

response_2045 = requests.get(url_prueba_2045, headers=headers, params=params)
print(response_2045.status_code)
data_2045 = response_2045.json()
print(len(data_2045['indicator']['values']))
print(set([v['geo_name'] for v in data_2045['indicator']['values']]))
data_2045['indicator']['values'][:3]

In [ ]:
def procesar_diccionario(diccionario):
    resultado_df={}
    for nombre in diccionario:
        valores = diccionario[nombre]['indicator']['values']
        resultado_df[nombre] = pd.DataFrame(valores)
        resultado_df[nombre] = resultado_df[nombre].rename(columns={'value': f'{nombre}'})
        # 👉 sobrescribe datetime_utc con el correcto, desde 'datetime' (que lleva el offset)
        resultado_df[nombre]['datetime_utc'] = pd.to_datetime(resultado_df[nombre]['datetime'], utc=True)
        resultado_df[nombre] = resultado_df[nombre][[nombre,'datetime_utc']]
    return resultado_df

<!-- ### Nota: verificación de cobertura temporal — ESIOS (demanda_prevista, id 544)

Al descargar el indicador `demanda_prevista` (id 544) para el rango completo 
2023-01-01 a 2026-06-30 con `time_trunc=hour`, se obtuvieron **30.622 registros**.

**Verificación:** se generó el rango horario teórico completo con `pd.date_range(freq='1h')` 
y se comparó contra las fechas presentes en la respuesta de la API (tras normalizar con 
`pd.to_datetime(..., utc=True)` para evitar el choque de offsets `+01:00`/`+02:00` propio 
de los cambios de horario). Se encontraron **4 horas "faltantes"**:

| Fecha | Causa |
|---|---|
| 2024-10-27 01:00:00 | Cambio de horario de invierno (última hora ambigua de octubre) |
| 2025-10-26 01:00:00 | Cambio de horario de invierno (última hora ambigua de octubre) |
| 2026-06-29 23:00:00 | Artefacto de borde en la conversión UTC ↔ hora local, en el límite final del rango solicitado |
| 2026-06-30 00:00:00 | Artefacto de borde en la conversión UTC ↔ hora local, en el límite final del rango solicitado |

**Interpretación:** 2 de los 4 huecos corresponden al fenómeno esperado y conocido del 
cambio de horario de otoño en España (hora local ambigua/duplicada). Los otros 2 son un 
efecto de los límites (`start_date`/`end_date`) de esta verificación puntual, no de la 
ingesta en sí.

**Decisión:** dado que representa un 0,013% del total de registros (4 de 30.622), 
no se considera necesario un tratamiento especial en esta fase. Se documenta como 
constancia de verificación de calidad de datos, siguiendo el mismo criterio aplicado a AEMET. -->


In [ ]:
diccionario_df = procesar_diccionario(diccionario)
diccionario_df_explicativo = procesar_diccionario(diccionario_explicativo)

In [ ]:
diccionario_df

In [ ]:
variable_acumulativa = list(diccionario_df.values())[0]

for df in list(diccionario_df)[1::]:
 variable_acumulativa = variable_acumulativa.merge(diccionario_df[df], on='datetime_utc', how='outer')

tabla_esios = variable_acumulativa

    


In [ ]:
tabla_esios
tabla_esios.duplicated().sum()

In [ ]:
variable_acumulativa_explicativo = list(diccionario_df_explicativo.values())[0]

for df in list(diccionario_df_explicativo)[1::]:
 variable_acumulativa_explicativo = variable_acumulativa_explicativo.merge(diccionario_df_explicativo[df], on='datetime_utc', how='outer')

tabla_esios_explicativo = variable_acumulativa_explicativo

In [ ]:
tabla_esios_explicativo
tabla_esios_explicativo.isna().sum()

# 0  (aplica el mismo fix a expl si no lo hiciste)


In [ ]:
tabla_esios.to_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_ESIOS", index=False, encoding="utf-8")
tabla_esios_explicativo.to_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_ESIOS_explicativo", index=False, encoding="utf-8")